# ONNX Model Feature Extraction

Extract per-layer feature vectors from ONNX models using **onnx-tool**:
- **f1 (MACs)**: Estimated multiply-accumulate operations
- **f2 (Weights)**: Number of learnable weight parameters
- **f3 (Activations)**: Size of output activation tensors (elements)

> **Note**: [Netron](https://netron.app/) is a visualizer only and has no Python API for programmatic extraction of these metrics. [onnx-tool](https://github.com/ThanatosShinji/onnx-tool) provides the profiling API used here.
>
> **Environment**: Use conda env `droidenv` (same as `onnx_conversion.ipynb`). Install onnx-tool if needed: `conda activate droidenv && pip install onnx-tool`
>
> **Prerequisite**: Export ONNX models first via `onnx_conversion.ipynb` (produces `fnet.onnx`, `cnet.onnx`, `update_core.onnx` in project root).

In [6]:
import os
import numpy as np

try:
    import onnx_tool
except ImportError:
    print("onnx-tool not found. Install in droidenv: pip install onnx-tool")
    raise

import pandas as pd

In [7]:
# Paths to ONNX models (from project root)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

ONNX_MODELS = {
    'fnet': os.path.join(PROJECT_ROOT, 'fnet.onnx'),
    'cnet': os.path.join(PROJECT_ROOT, 'cnet.onnx'),
    'update_core': os.path.join(PROJECT_ROOT, 'update_core.onnx'),
}

# Optional: add custom paths
# ONNX_MODELS['my_model'] = '/path/to/model.onnx'

# Input shapes for shape inference (dynamic axes need concrete values)
# fnet/cnet: [batch, frames, 3, H, W]; update: [B, K, C, H/8, W/8]
DEFAULT_INPUTS = {
    'fnet': {'images': np.zeros((1, 3, 3, 240, 320), dtype=np.float32)},
    'cnet': {'images': np.zeros((1, 3, 3, 240, 320), dtype=np.float32)},
    'update_core': {
        'net': np.zeros((1, 3, 128, 30, 40), dtype=np.float32),
        'inp': np.zeros((1, 3, 128, 30, 40), dtype=np.float32),
        'corr': np.zeros((1, 3, 196, 30, 40), dtype=np.float32),
        'flow': np.zeros((1, 3, 4, 30, 40), dtype=np.float32),
    },
}

In [8]:
def profile_onnx_model(model_path: str, input_shapes: dict = None) -> pd.DataFrame:
    """
    Profile an ONNX model and return a DataFrame with per-layer:
    - layer: node name
    - MACs: multiply-accumulate operations
    - Weights: number of learnable parameters
    - Activations: output tensor size (elements)
    """
    if not os.path.exists(model_path):
        return pd.DataFrame(columns=['layer', 'MACs', 'Weights', 'Activations'])

    m = onnx_tool.Model(model_path)
    g = m.graph
    g.graph_reorder_nodes()

    if input_shapes:
        g.shape_infer(input_shapes)
    else:
        g.shape_infer()

    g.profile()

    def _to_int(x):
        """Convert macs/params to int; onnx-tool may return lists (e.g. [forward, backward])."""
        if x is None:
            return 0
        if isinstance(x, (list, tuple)):
            return int(x[0]) if x else 0
        return int(x)

    rows = []
    for name, node in g.nodemap.items():
        macs = _to_int(getattr(node, 'macs', None) or getattr(node, 'MACs', 0))
        params = _to_int(getattr(node, 'params', None) or getattr(node, 'n_param', None) or getattr(node, 'param_count', 0))

        # Activations = output tensor size (elements)
        activations = 0
        for out_name in node.output:
            if out_name in g.tensormap:
                t = g.tensormap[out_name]
                shape = getattr(t, 'shape', None)
                if shape is None and hasattr(t, 'get_shape'):
                    shape = t.get_shape()
                if shape is None and hasattr(t, 'dims'):
                    shape = list(t.dims) if t.dims else None
                if shape:
                    vol = 1
                    for d in shape:
                        if isinstance(d, (int, np.integer)) and d > 0:
                            vol *= int(d)
                        else:
                            vol = 0
                            break
                    activations += vol

        rows.append({
            'layer': name,
            'MACs': macs,
            'Weights': params,
            'Activations': activations,
        })

    return pd.DataFrame(rows)

In [9]:
# Run cells above, then execute the cell below to profile all models

In [10]:
# Run profiling for each model and display results
for model_name, model_path in ONNX_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({model_path})")
    print('='*60)

    inputs = DEFAULT_INPUTS.get(model_name)
    df = profile_onnx_model(model_path, inputs)

    if df.empty:
        print(f"  (model not found or no nodes)")
        continue

    display(df)
    print(f"\nTotal MACs: {df['MACs'].sum():,}")
    print(f"Total Weights: {df['Weights'].sum():,}")
    print(f"Total Activations (sum): {df['Activations'].sum():,}")


Model: fnet (/home/campus.ncl.ac.uk/c4071391/Projects/DROID-SLAM/fnet.onnx)


,layer,MACs,Weights,Activations
0,Constant_49,0,0,3
1,/norm/Constant,0,0,0
2,/fnet/Constant,0,0,0
3,/fnet/Constant_1,0,0,0
4,/fnet/Constant_2,0,0,0
...,...,...,...,...
134,/fnet/Unsqueeze_6,0,0,1
135,/fnet/Unsqueeze_7,0,0,1
136,/fnet/Unsqueeze_8,0,0,1
137,/fnet/Concat_1,0,0,5



Total MACs: 6,391,526,400
Total Weights: 714,566
Total Activations (sum): 64,283,907

Model: cnet (/home/campus.ncl.ac.uk/c4071391/Projects/DROID-SLAM/cnet.onnx)


,layer,MACs,Weights,Activations
0,Constant_20,0,0,3
1,/norm/Constant,0,0,0
2,/cnet/Constant,0,0,0
3,/cnet/Constant_1,0,0,0
4,/cnet/Constant_2,0,0,0
...,...,...,...,...
93,/cnet/Concat_1,0,0,5
94,/cnet/Reshape_1,0,0,921600
95,/Split,0,0,921600
96,/Tanh,35020800,0,460800



Total MACs: 6,373,555,200
Total Weights: 731,078
Total Activations (sum): 50,918,469

Model: update_core (/home/campus.ncl.ac.uk/c4071391/Projects/DROID-SLAM/update_core.onnx)


,layer,MACs,Weights,Activations
0,/update/Shape,0,0,5
1,/update/Constant,0,0,0
2,/update/Shape_1,0,0,5
3,/update/Constant_1,0,0,0
4,/update/Shape_2,0,0,5
...,...,...,...,...
162,/update/Reshape_5,0,0,7200
163,/update/Transpose,0,0,7200
164,/update/Slice,0,0,7200
165,/update/Transpose_1,0,0,7200



Total MACs: 7,779,158,208
Total Weights: 2,186,308
Total Activations (sum): 19,924,477
